# Steam Sales Dataset Analysis (LLM Fine-Tuning)

This section focuses on fine-tuning a frontier Large Language Model (LLM) using a curated subset of the Steam dataset to improve task-specific performance. While previous stages explored preprocessing, baseline models, and deep learning approaches, fine-tuning allows us to adapt a powerful pre-trained model to better understand domain-specific patterns within game sales related data.

The primary objective of this phase is to enhance the model’s ability to capture nuanced relationships between textual descriptions, metadata, and pricing by leveraging transfer learning. Instead of training a model from scratch, fine-tuning builds on an already knowledgeable model and specializes it for the Steam marketplace context.

## Import Libraries

In [18]:
import os
import re
import json
from dotenv import load_dotenv
from pathlib import Path

from huggingface_hub import login
from openai import OpenAI

from sales_util.evaluator import evaluate
from sales_util.items_data import Item

In [2]:
# Load env file
load_dotenv(override=True)

True

In [19]:
BASE_DIR = Path(os.getenv("PROJECT_ROOT"))

## Load Dataset From HuggingFace

In [3]:
username = "KumudithaSilva"
dataset = f"{username}/items_llm_raw_lite"

train, val, test = Item.from_hub(dataset)

items = train + val + test

print(f"Loaded {len(items):,} items")

README.md: 0.00B [00:00, ?B/s]

c:\Users\REDTECH\miniconda3\envs\ml-dl-fine-tuning\Lib\site-packages\huggingface_hub\file_download.py:138: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\REDTECH\.cache\huggingface\hub\datasets--KumudithaSilva--items_llm_raw_lite. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


data/train-00000-of-00001.parquet:   0%|          | 0.00/1.36M [00:00<?, ?B/s]

data/validation-00000-of-00001.parquet:   0%|          | 0.00/181k [00:00<?, ?B/s]

data/test-00000-of-00001.parquet:   0%|          | 0.00/182k [00:00<?, ?B/s]

Generating train split:   0%|          | 0/17600 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/2200 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/2200 [00:00<?, ? examples/s]

Loaded 22,000 items


## Fine-tuning Dataset

In [ ]:
fine_tune_train = train[:100]
fine_tune_validation = val[:50]

## Fine-tuning Prompt

In [ ]:
def format_item(item):
    return f"""
Game: {item.name}
Peak CCU: {item.peakCCU}
Required Age: {item.required_age}
DLC Count: {item.dlcCount}
Supports Windows: {item.supportWindows}
Supports Mac: {item.supportMac}
Supports Linux: {item.supportLinux}
Positive Reviews: {item.positive}
Negative Reviews: {item.negative}
Achievements: {item.achievements}
Recommendations: {item.recommendations}
Release Date: {item.release_year}-{item.release_month}-{item.release_day}
Estimated Owners: {item.min_estimatedOwners} - {item.max_estimatedOwners}
Languages Supported: {item.supported_languages}
Developers: {item.num_developers}
Publishers: {item.num_publishers}
Categories: {item.num_categories}
Genres: {item.num_genres}
Description: {item.small_description}
"""

In [11]:
def build_prompt(item):
    return f"Estimate the price of this game. Respond with only the price.\n\n{format_item(item)}"

In [13]:
def messages_for(item):
    return [
        {"role": "user", "content": build_prompt(item)},
        {"role": "assistant", "content": f"${item.price:.2f}"}
    ]

In [14]:
def test_messages_for(item):
    return [
        {"role": "user", "content": build_prompt(item)},
    ]

##  JSONL Files

In [20]:
JSONL_PATH = BASE_DIR / "data"

In [21]:
def make_jsonl(items):
    result = ""
    for item in items:
        messages = messages_for(item)
        messages_str = json.dumps(messages)
        result += '{"messages": ' + messages_str +'}\n'
    return result.strip()

In [23]:
def write_jsonl(items, filename):
    with open(filename, "w") as f:
        jsonl = make_jsonl(items)
        f.write(jsonl)

In [24]:
write_jsonl(fine_tune_train, f"{JSONL_PATH}/fine_tune_train.jsonl")

In [26]:
write_jsonl(fine_tune_validation, f"{JSONL_PATH}/fine_tune_validation.jsonl")

## Fine-tuning Mode

In [25]:
openai = OpenAI(api_key=os.environ.get("OPENAI_API_KEY"))

In [27]:
with open(f"{JSONL_PATH}/fine_tune_train.jsonl", "rb") as f:
    train_file = openai.files.create(file=f, purpose="fine-tune")
train_file

FileObject(id='file-3DT8SFcvcEGWgirnaT4qYE', bytes=57924, created_at=1776870607, filename='fine_tune_train.jsonl', object='file', purpose='fine-tune', status='processed', expires_at=None, status_details=None)

In [ ]:
train_file.id

'file-3DT8SFcvcEGWgirnaT4qYE'

In [28]:
with open(f"{JSONL_PATH}/fine_tune_validation.jsonl", "rb") as f:
    validation_file = openai.files.create(file=f, purpose="fine-tune")
validation_file

FileObject(id='file-RwjBno9NVf7rru3vCqLL12', bytes=29307, created_at=1776870632, filename='fine_tune_validation.jsonl', object='file', purpose='fine-tune', status='processed', expires_at=None, status_details=None)

In [30]:
validation_file.id

'file-RwjBno9NVf7rru3vCqLL12'

## Supervised fine-tuning

In [31]:
openai.fine_tuning.jobs.create(
    training_file=train_file.id,
    validation_file=validation_file.id,
    model="gpt-4.1-nano-2025-04-14",
    seed=42,
    hyperparameters={"n_epochs": 1, "batch_size": 1},
    suffix="pricer"
)

FineTuningJob(id='ftjob-4tzM7runna2FOH5SIvgwjsXj', created_at=1776871230, error=Error(code=None, message=None, param=None), fine_tuned_model=None, finished_at=None, hyperparameters=Hyperparameters(batch_size=1, learning_rate_multiplier='auto', n_epochs=1), model='gpt-4.1-nano-2025-04-14', object='fine_tuning.job', organization_id='org-xjW5t7Ao8BqqBd4Evyu9xjyG', result_files=[], seed=42, status='validating_files', trained_tokens=None, training_file='file-3DT8SFcvcEGWgirnaT4qYE', validation_file='file-RwjBno9NVf7rru3vCqLL12', estimated_finish=None, integrations=[], metadata=None, method=Method(type='supervised', dpo=None, reinforcement=None, supervised=SupervisedMethod(hyperparameters=SupervisedHyperparameters(batch_size=1, learning_rate_multiplier='auto', n_epochs=1))), user_provided_suffix='pricer', usage_metrics=None, shared_with_openai=False, eval_id=None, internal_worker_backend=None, internal_peashooter_execution=None, train_experiment_id=None, eval_experiment_id=None)

## Fine-tunning List

In [32]:
openai.fine_tuning.jobs.list(limit=1)

SyncCursorPage[FineTuningJob](data=[FineTuningJob(id='ftjob-4tzM7runna2FOH5SIvgwjsXj', created_at=1776871230, error=Error(code=None, message=None, param=None), fine_tuned_model=None, finished_at=None, hyperparameters=Hyperparameters(batch_size=1, learning_rate_multiplier=0.1, n_epochs=1), model='gpt-4.1-nano-2025-04-14', object='fine_tuning.job', organization_id='org-xjW5t7Ao8BqqBd4Evyu9xjyG', result_files=[], seed=42, status='validating_files', trained_tokens=None, training_file='file-3DT8SFcvcEGWgirnaT4qYE', validation_file='file-RwjBno9NVf7rru3vCqLL12', estimated_finish=None, integrations=[], metadata=None, method=Method(type='supervised', dpo=None, reinforcement=None, supervised=SupervisedMethod(hyperparameters=SupervisedHyperparameters(batch_size=1, learning_rate_multiplier=0.1, n_epochs=1))), user_provided_suffix='pricer', usage_metrics=None, shared_with_openai=False, eval_id=None, internal_worker_backend=None, internal_peashooter_execution=None, train_experiment_id=None, eval_ex

In [34]:
job_id = openai.fine_tuning.jobs.list(limit=1).data[0].id
job_id

'ftjob-4tzM7runna2FOH5SIvgwjsXj'

In [35]:
openai.fine_tuning.jobs.retrieve(job_id)

FineTuningJob(id='ftjob-4tzM7runna2FOH5SIvgwjsXj', created_at=1776871230, error=Error(code=None, message=None, param=None), fine_tuned_model=None, finished_at=None, hyperparameters=Hyperparameters(batch_size=1, learning_rate_multiplier=0.1, n_epochs=1), model='gpt-4.1-nano-2025-04-14', object='fine_tuning.job', organization_id='org-xjW5t7Ao8BqqBd4Evyu9xjyG', result_files=[], seed=42, status='queued', trained_tokens=None, training_file='file-3DT8SFcvcEGWgirnaT4qYE', validation_file='file-RwjBno9NVf7rru3vCqLL12', estimated_finish=None, integrations=[], metadata=None, method=Method(type='supervised', dpo=None, reinforcement=None, supervised=SupervisedMethod(hyperparameters=SupervisedHyperparameters(batch_size=1, learning_rate_multiplier=0.1, n_epochs=1))), user_provided_suffix='pricer', usage_metrics=None, shared_with_openai=False, eval_id=None, internal_worker_backend=None, internal_peashooter_execution=None, train_experiment_id=None, eval_experiment_id=None)

In [41]:
openai.fine_tuning.jobs.list_events(fine_tuning_job_id=job_id, limit=10).data

[FineTuningJobEvent(id='ftevent-yWENPktXUlNmweX7nu4k8r15', created_at=1776871332, level='info', message='Files validated, moving job to queued state', object='fine_tuning.job.event', data={}, type='message'),
 FineTuningJobEvent(id='ftevent-mcEMxInSYA6YYzVonhSxMpDQ', created_at=1776871230, level='info', message='Validating training file: file-3DT8SFcvcEGWgirnaT4qYE and validation file: file-RwjBno9NVf7rru3vCqLL12', object='fine_tuning.job.event', data={}, type='message'),
 FineTuningJobEvent(id='ftevent-qY1u1NyhR7rRwM6w68pqauG7', created_at=1776871230, level='info', message='Created fine-tuning job: ftjob-4tzM7runna2FOH5SIvgwjsXj', object='fine_tuning.job.event', data={}, type='message')]

## Fine-tunning Model Test

In [54]:
fine_tuned_model_name = openai.fine_tuning.jobs.retrieve(job_id).fine_tuned_model
fine_tuned_model_name

In [62]:
fine_tuned_status = openai.fine_tuning.jobs.retrieve(job_id).status

In [46]:
def gpt_4__1_nano_fine_tuned(item):
    response = openai.chat.completions.create(
        model=fine_tuned_model_name,
        messages=test_messages_for(item),
        max_tokens=7
    )
    return response.choices[0].message.content

In [ ]:
print(test[0].price)
print(gpt_4__1_nano_fine_tuned(test[0]))

In [63]:
if fine_tuned_status == "succeeded":
    print(test[0].price)
    print(gpt_4__1_nano_fine_tuned(test[0]))
else:
    print("Fine-tuning in progress")

Fine-tuning in progress


In [ ]:
evaluate(gpt_4__1_nano_fine_tuned, test)